# Digital Mirror v0.10: does the AI improve itself?

**How to run.** *Runtime → Change runtime type →* pick a **GPU** (T4 works; L4 or A100 is faster) → *Save*.
Then *Runtime → **Run all***. Keep this tab open. The last cell downloads `…_evidence.zip`:
upload that file to Claude.

What the cells do, in order:
1. verify the embedded, hash-checked source archive and run the software tests;
2. detect the GPU, install the pinned inference libraries, and test the real
   inference stack on the GPU with a tiny local model;
3. run a short preflight with the real model (Qwen2.5-3B-Instruct, pinned
   revision) and print protocol diagnostics;
4. run the experiment: for each of several independent task plans, the agent
   recursively examines and rewrites its own strategy and self-model, keeping
   only changes that pass paired tests on fresh tasks; then a held-out final
   test. A control arm repeats this from a hand-written step-by-step prompt;
5. pool the held-out results across plans and export all evidence.

Expected time with the default settings: about 2 hours on a T4, less on L4/A100.
Use the settings cell to shorten it (for example fewer `SEEDS`).
If any cell stops with an error, run the **last cell** by hand: it exports
everything produced so far, including logs of the failure.

A positive result shows measured, transferable improvement of this model's
inference-time strategy or self-model on these task types. It does not show
general self-improvement, changed weights, or subjective awareness.


In [ ]:
import base64, hashlib, io, json, os, stat, sys, uuid, zipfile
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
SOURCE_ZIP_SHA256 = "e269b8e473bb1ee9810da825d3e9a3298fee1562eb6cce2c7f44e6ff54570525"
blob = base64.b64decode("...", validate=True)
assert hashlib.sha256(blob).hexdigest() == SOURCE_ZIP_SHA256, "Source archive mismatch"
z = zipfile.ZipFile(io.BytesIO(blob))
infos = z.infolist()
names = [i.filename for i in infos]
assert len(names) == len(set(names)) and len(names) < 100, "Unexpected archive inventory"
assert sum(i.file_size for i in infos) < 4_000_000, "Archive exceeds source size bound"
for item in infos:
    path = PurePosixPath(item.filename)
    assert not path.is_absolute() and ".." not in path.parts and ":" not in item.filename
    assert "\\" not in item.filename and not item.is_dir()
    assert stat.S_IFMT(item.external_attr >> 16) == stat.S_IFREG, "Nonregular archive member"
data = {name: z.read(name) for name in names}
manifest = json.loads(data["MANIFEST.json"])
assert manifest["format"] == "recursive-mirror-release-v1"
assert set(data) == set(manifest["files"]) | {"MANIFEST.json"}
for name, meta in manifest["files"].items():
    assert len(data[name]) == meta["bytes"]
    assert hashlib.sha256(data[name]).hexdigest() == meta["sha256"], name
SESSION = Path("/content") / ("recursive_mirror_v0100_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8])
SESSION.mkdir(exist_ok=False)
PROJECT = SESSION / "project"
PROJECT.mkdir()
for name, content in data.items():
    target = PROJECT.joinpath(*PurePosixPath(name).parts)
    target.parent.mkdir(parents=True, exist_ok=True)
    with target.open("xb") as handle:
        handle.write(content)
with (SESSION / "source.zip").open("xb") as handle:
    handle.write(blob)
os.chdir(PROJECT)
print(json.dumps({"event": "source_verified", "session": str(SESSION), "source_sha256": SOURCE_ZIP_SHA256, "files": len(data)}))


In [ ]:
# Experiment settings: edit before "Run all" for a shorter or longer run.
MODEL = "Qwen/Qwen2.5-3B-Instruct"
REVISION = "aa8e72537993ba99e69dfaafa59ed015b17504d1"  # pinned commit of MODEL; change both together
SEEDS = "2026,2027,2028"      # independent task plans; fewer seeds = shorter run, weaker evidence
RUN_CONTROL = True            # also start from a hand-written step-by-step prompt
ROUNDS, DISCOVERY, VALIDATION, FINAL = 3, 12, 24, 48   # larger validation/final sets = more power
MAX_CALLS_PER_RUN = 720       # 3 levels need at most 711 calls with these sizes
TOTAL_HOURS = 4.0             # wall-clock budget shared by all runs of the main experiment
FORCE_AFTER_WEAK_PREFLIGHT = False  # True: run the experiment even if the preflight looks uninformative
print({"model": MODEL, "seeds": SEEDS, "control": RUN_CONTROL, "hours": TOTAL_HOURS})


In [ ]:
import subprocess, time, shutil
sys.path.insert(0, str(PROJECT))
ENV = dict(os.environ, HF_HOME="/content/hf_cache", USE_TF="0", TOKENIZERS_PARALLELISM="false")

def captured(label, command, timeout=120, env=None):
    print("Starting:", label, flush=True)
    log = SESSION / (label + ".log")
    # Direct file descriptors preserve partial output even if the cell
    # is interrupted before the child exits. Never overwrite a log.
    handle = log.open("x", encoding="utf-8")
    try:
        completed = subprocess.run(command, cwd=PROJECT, stdout=handle, env=env or ENV,
                                   stderr=subprocess.STDOUT, text=True, timeout=timeout)
    finally:
        handle.flush()
        os.fsync(handle.fileno())
        handle.close()
        print(log.read_text(encoding="utf-8", errors="replace"))
    if completed.returncode:
        raise RuntimeError(f"{label} exited {completed.returncode}; log preserved in {SESSION}")
    return completed

def streamed(label, command, timeout):
    """Run a long child process and show its log live; returns the exit code."""
    print("Starting:", label, flush=True)
    log = SESSION / (label + ".log")
    handle = log.open("x", encoding="utf-8")
    process = subprocess.Popen(command, cwd=PROJECT, stdout=handle, stderr=subprocess.STDOUT,
                               text=True, env=ENV)
    shown, deadline = 0, time.monotonic() + timeout
    try:
        while process.poll() is None:
            time.sleep(10)
            text = log.read_text(encoding="utf-8", errors="replace")
            print(text[shown:], end="", flush=True)
            shown = len(text)
            if time.monotonic() > deadline:
                print(f"\n{label}: notebook timeout reached; stopping the child", flush=True)
                process.terminate()
                process.wait(120)
    except KeyboardInterrupt:
        process.terminate()
        process.wait(120)
        raise
    finally:
        handle.flush()
        os.fsync(handle.fileno())
        handle.close()
        text = log.read_text(encoding="utf-8", errors="replace")
        print(text[shown:], flush=True)
    return process.returncode

def export_evidence():
    """Zip the whole session (sources, logs, journals, results) and offer it for download."""
    stamp = datetime.now(timezone.utc).strftime("%H%M%S")
    export = Path("/content") / f"{SESSION.name}_evidence_{stamp}.zip"
    with zipfile.ZipFile(export, "x", compression=zipfile.ZIP_DEFLATED) as bundle:
        for path in sorted(SESSION.rglob("*")):
            if path.is_symlink():
                raise RuntimeError("Unexpected symlink in evidence session")
            if path.is_file():
                bundle.write(path, path.relative_to(SESSION).as_posix())
    digest = hashlib.sha256(export.read_bytes()).hexdigest()
    print(json.dumps({"event": "evidence_export", "path": str(export), "bytes": export.stat().st_size,
                      "sha256": digest}))
    try:
        from google.colab import files
        files.download(str(export))
    except ImportError:
        print("Not running in Colab: download", export, "manually.")
    return export


In [ ]:
captured("unit-tests", [sys.executable, "-B", "-m", "unittest", "discover", "-s", "tests", "-v"])
captured("scripted-demo", [sys.executable, "-B", "-m", "recursive_mirror", "demo", "--output",
                           str(SESSION / "scripted_demo")])
print("Software checks complete. The scripted demo is NOT an LLM experiment.")


## GPU, pinned libraries and a real-stack test

Checks for a GPU before installing anything. The inference libraries are pinned
(`transformers==4.57.6`). A tiny locally built Qwen2-architecture model then
exercises the real backend on the GPU (generation, batching, EOS, truncation)
in about a minute, before any long run.


In [ ]:
captured("gpu-status", ["nvidia-smi"], timeout=60)
import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > GPU, then Run all again."
GPU = torch.cuda.get_device_name(0)
CAPABILITY = torch.cuda.get_device_capability(0)
DTYPE = "bfloat16" if CAPABILITY[0] >= 8 else "float16"   # T4 (7.5) has native FP16 only
MEMORY_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
BATCH = 16 if MEMORY_GB >= 30 else 8
captured("inference-dependencies", [sys.executable, "-m", "pip", "install", "-q",
         "transformers==4.57.6", "accelerate>=0.33,<2"], timeout=900)
from importlib.metadata import version
runtime_info = {"python": sys.version, "gpu": GPU, "compute_capability": list(CAPABILITY),
                "memory_gb": round(MEMORY_GB, 1), "dtype": DTYPE, "batch_size": BATCH,
                "versions": {name: version(name) for name in ["torch", "transformers", "accelerate"]}}
(SESSION / "runtime-info.json").write_text(json.dumps(runtime_info, indent=2), encoding="utf-8")
print(json.dumps(runtime_info, indent=2))
# Load the test by file path: Colab ships an unrelated top-level package named tests,
# which shadows this project's tests folder when a test is imported by dotted name.
captured("gpu-integration-test", [sys.executable, "-B", "-m", "unittest", "discover", "-s", "tests",
         "-p", "test_hf_integration.py", "-v"], timeout=900, env=dict(ENV, RECURSIVE_MIRROR_TEST_DEVICE="cuda"))
MODEL_ARGS = ["--model", MODEL, "--revision", REVISION, "--allow-download", "--device", "cuda",
              "--dtype", DTYPE, "--batch-size", str(BATCH), "--max-input-tokens", "8192"]


## Preflight with the real model (about 5 minutes)

Downloads the pinned model once, then runs one short level on a separate small
plan and prints diagnostics: answer-line use, invalid forecasts, truncations and
speed. It stops before the long run only if the model cannot follow the protocol
at all.


In [ ]:
from recursive_mirror.diagnostics import diagnose, diagnosis_markdown
PREFLIGHT_PLAN = SESSION / "preflight-plan.json"
PREFLIGHT = SESSION / "preflight"
captured("preflight-plan", [sys.executable, "-B", "-m", "recursive_mirror", "make-plan", "--output",
         str(PREFLIGHT_PLAN), "--seed", "7", "--rounds", "1", "--discovery", "6", "--validation", "3",
         "--final", "3"])
code = streamed("preflight-run", [sys.executable, "-B", "-m", "recursive_mirror", "run", "--plan",
                str(PREFLIGHT_PLAN), "--output", str(PREFLIGHT), *MODEL_ARGS, "--max-depth", "1",
                "--max-calls", "60"], timeout=3600)
health = diagnose(PREFLIGHT)
print(diagnosis_markdown(PREFLIGHT, health))
answers = sum(health["answer_status"].values()) or 1
usable = sum(health["answer_status"].get(k, 0) for k in ("answer_line", "boxed", "last_line"))
valid = health["forecasts"].get("valid", 0) / (sum(health["forecasts"].values()) or 1)
problems = []
if code != 0:
    problems.append(f"the preflight run exited with code {code}")
if usable / answers < 0.5:
    problems.append(f"only {usable}/{answers} answers were usable (the rest hit the token limit or were empty)")
if valid < 0.5:
    problems.append(f"only {valid:.0%} of forecasts were valid percentages")
if problems and not FORCE_AFTER_WEAK_PREFLIGHT:
    raise RuntimeError("Preflight looks uninformative: " + "; ".join(problems) + ". Run the last cell to "
                       "export the evidence and send it to Claude (or set FORCE_AFTER_WEAK_PREFLIGHT).")
print("Preflight passed" + (" with warnings: " + "; ".join(health["warnings"] + problems)
                            if health["warnings"] or problems else "."))


## The experiment

Each run writes its own journal, result, checkpoint and `report.md`; a pooled
`summary.md` is rewritten after every run, so even an interrupted session keeps
usable evidence. Progress lines appear below as levels complete.


In [ ]:
EXPERIMENT = SESSION / "experiment"
TOTAL_SECONDS = int(TOTAL_HOURS * 3600)  # shared out as it is spent: fast runs leave time for later ones
command = [sys.executable, "-B", "-m", "recursive_mirror", "experiment", "--output", str(EXPERIMENT),
           *MODEL_ARGS, "--seeds", SEEDS, "--rounds", str(ROUNDS), "--discovery", str(DISCOVERY),
           "--validation", str(VALIDATION), "--final", str(FINAL), "--max-depth", str(ROUNDS),
           "--max-calls", str(MAX_CALLS_PER_RUN), "--total-seconds", str(TOTAL_SECONDS)]
if not RUN_CONTROL:
    command += ["--control-strategy", ""]
code = streamed("experiment", command, timeout=int(TOTAL_HOURS * 3600) + 3600)
summary = EXPERIMENT / "summary.md"
print(summary.read_text(encoding="utf-8") if summary.exists() else "No summary was written; see experiment.log")
if code:
    print("Some runs did not complete; their evidence is still exported by the next cell.")


## Export evidence (also run this by hand after any error)

Packages this session's sources, logs, journals, results and summaries into one
zip and downloads it. Upload the zip to Claude for analysis.


In [ ]:
EXPORT = export_evidence()
